# 最终版第一版

因为发现上面长时间用填充数据不好，官网显示数据不具备参考价格，所以准备全部重新获取raw data, 然后再格子统一颗粒度
同时发现部分国家不存在电价，所以直接不进入统计。
这一版就是纯XMl数据，没有任何修饰处理的，直接从官网下载

In [ ]:
import os
import pandas as pd

from entsoe import EntsoeRawClient
from entsoe.mappings import NEIGHBOURS


# =========================
# 基本参数
# =========================
API_KEY = "7a7fb42a-192c-4bae-abf0-e0d6d3810018"
TZ = "UTC"

START = pd.Timestamp("2024-01-01 00:00", tz=TZ)
END = pd.Timestamp("2026-01-01 00:00", tz=TZ)

raw_client = EntsoeRawClient(api_key=API_KEY)


# =========================
# 保存 XML
# =========================
def save_xml(text, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)


# =========================
# 下载单个国家
# 规则：
# 1. 先检查 price 是否存在
# 2. 如果没有 price，直接跳过，不创建文件夹
# 3. 如果有 price，再创建文件夹并保存全部 XML
# =========================
def download_one_country(country):
    print("\n==============================")
    print("Processing:", country)
    print("==============================")

    neighbours = NEIGHBOURS[country]

    # =========================
    # 1. 先检查 PRICE（不创建文件夹）
    # =========================
    try:
        print("Checking PRICE XML...")
        xml_price = raw_client.query_day_ahead_prices(country, START, END)
        print("Price exists -> continue")
    except Exception as e:
        print("Skip country %s: no price or price download failed: %s" % (country, e))
        return

    # =========================
    # 2. 只有 price 存在才创建文件夹
    # =========================
    base_path = country
    raw_xml_path = os.path.join(base_path, "raw_xml")

    os.makedirs(base_path, exist_ok=True)
    os.makedirs(raw_xml_path, exist_ok=True)

    # =========================
    # 3. 保存 PRICE
    # =========================
    try:
        save_xml(xml_price, os.path.join(raw_xml_path, "%s_price.xml" % country))
        print("Saved PRICE")
    except Exception as e:
        print("Failed to save PRICE for %s: %s" % (country, e))
        return

    # =========================
    # 4. LOAD
    # =========================
    try:
        print("Downloading LOAD XML...")
        xml_load = raw_client.query_load(country, START, END)
        save_xml(xml_load, os.path.join(raw_xml_path, "%s_load.xml" % country))
        print("Finish LOAD")
    except Exception as e:
        print("Skip LOAD for %s: %s" % (country, e))

    # =========================
    # 5. GENERATION
    # =========================
    try:
        print("Downloading GENERATION XML...")
        xml_generation = raw_client.query_generation(country, START, END)
        save_xml(xml_generation, os.path.join(raw_xml_path, "%s_generation.xml" % country))
        print("Finish GENERATION")
    except Exception as e:
        print("Skip GENERATION for %s: %s" % (country, e))

    # =========================
    # 6. CROSSBORDER
    # =========================
    print("Downloading CROSSBORDER XML...")

    for nb in neighbours:
        # export: country -> nb
        try:
            xml_export = raw_client.query_crossborder_flows(country, nb, START, END)
            save_xml(
                xml_export,
                os.path.join(raw_xml_path, "%s_to_%s.xml" % (country, nb))
            )
            print("Finish export:", country, "->", nb)
        except Exception as e:
            print("Skip export %s -> %s: %s" % (country, nb, e))

        # import: nb -> country
        try:
            xml_import = raw_client.query_crossborder_flows(nb, country, START, END)
            save_xml(
                xml_import,
                os.path.join(raw_xml_path, "%s_to_%s.xml" % (nb, country))
            )
            print("Finish import:", nb, "->", country)
        except Exception as e:
            print("Skip import %s -> %s: %s" % (nb, country, e))

    print("Finished:", country)


# =========================
# 主循环：遍历所有国家
# 只用官方国家列表
# =========================
def main():
    all_countries = sorted(NEIGHBOURS.keys())

    for country in all_countries:
        download_one_country(country)

    print("\nALL COUNTRIES DONE.")


if __name__ == "__main__":
    main()

读取下载的文件，然后转成CSV

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

from entsoe.mappings import NEIGHBOURS


# =========================
# 工具函数
# =========================
def get_namespace(root):
    if root.tag.startswith("{"):
        return root.tag.split("}")[0][1:]
    return ""


def qname(ns, tag):
    if ns:
        return "{%s}%s" % (ns, tag)
    return tag


def resolution_to_freq(resolution):
    mapping = {
        "PT15M": "15min",
        "PT30M": "30min",
        "PT60M": "1h",
        "PT1H": "1h",
    }
    if resolution not in mapping:
        raise ValueError("Unsupported resolution: %s" % resolution)
    return mapping[resolution]


# =========================
# 最正常、完整的 XML -> CSV 转化
# 一行对应一个 Point
# 保留这个 Point 对应的完整上下文信息
# 不补值、不聚合、不筛选
# =========================
def parse_xml_generic(xml_path, csv_path, value_tag, value_col_name):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = get_namespace(root)

    rows = []

    for ts in root.findall(".//" + qname(ns, "TimeSeries")):
        business_type = ts.findtext(qname(ns, "businessType"), default="")
        in_domain = ts.findtext(qname(ns, "in_Domain.mRID"), default="")
        out_domain = ts.findtext(qname(ns, "out_Domain.mRID"), default="")

        psr_type = ""
        mkt_psr = ts.find(qname(ns, "MktPSRType"))
        if mkt_psr is not None:
            psr_elem = mkt_psr.find(qname(ns, "psrType"))
            if psr_elem is not None and psr_elem.text is not None:
                psr_type = psr_elem.text

        currency_unit = ts.findtext(qname(ns, "currency_Unit.name"), default="")
        price_measure_unit = ts.findtext(qname(ns, "price_Measure_Unit.name"), default="")

        for period in ts.findall(qname(ns, "Period")):
            ti = period.find(qname(ns, "timeInterval"))
            if ti is None:
                continue

            start_elem = ti.find(qname(ns, "start"))
            end_elem = ti.find(qname(ns, "end"))
            res_elem = period.find(qname(ns, "resolution"))

            if start_elem is None or end_elem is None or res_elem is None:
                continue

            start_text = start_elem.text
            end_text = end_elem.text
            resolution = res_elem.text
            freq = resolution_to_freq(resolution)

            idx = pd.date_range(
                start=start_text,
                end=end_text,
                freq=freq,
                inclusive="left",
                tz="UTC"
            )

            for point in period.findall(qname(ns, "Point")):
                pos_elem = point.find(qname(ns, "position"))
                val_elem = point.find(qname(ns, value_tag))

                if pos_elem is None or val_elem is None:
                    continue
                if val_elem.text in (None, ""):
                    continue

                position = int(pos_elem.text)
                value = float(val_elem.text)

                if 1 <= position <= len(idx):
                    rows.append({
                        "datetime": idx[position - 1],
                        value_col_name: value,
                        "position": position,
                        "resolution": resolution,
                        "business_type": business_type,
                        "psr_type": psr_type,
                        "in_domain": in_domain,
                        "out_domain": out_domain,
                        "currency_unit": currency_unit,
                        "price_measure_unit": price_measure_unit
                    })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")


# =========================
# 单个国家转换
# 原目录：
# ./{country}/raw_xml/
#
# 输出目录：
# ./{country}/csv/
# =========================
def convert_one_country(country):
    print("\n==============================")
    print("Processing:", country)
    print("==============================")

    base_path = country
    raw_xml_path = os.path.join(base_path, "raw_xml")
    csv_path = os.path.join(base_path, "csv")

    if not os.path.exists(raw_xml_path):
        print("Skip %s: raw_xml folder not found" % country)
        return

    os.makedirs(csv_path, exist_ok=True)

    # =========================
    # 1. PRICE
    # =========================
    price_xml = os.path.join(raw_xml_path, "%s_price.xml" % country)
    if os.path.exists(price_xml):
        try:
            print("Converting PRICE XML -> CSV...")
            parse_xml_generic(
                price_xml,
                os.path.join(csv_path, "%s_price.csv" % country),
                "price.amount",
                "price"
            )
            print("Finish PRICE")
        except Exception as e:
            print("Failed PRICE for %s: %s" % (country, e))
    else:
        print("PRICE XML not found for %s" % country)

    # =========================
    # 2. LOAD
    # =========================
    load_xml = os.path.join(raw_xml_path, "%s_load.xml" % country)
    if os.path.exists(load_xml):
        try:
            print("Converting LOAD XML -> CSV...")
            parse_xml_generic(
                load_xml,
                os.path.join(csv_path, "%s_load.csv" % country),
                "quantity",
                "quantity"
            )
            print("Finish LOAD")
        except Exception as e:
            print("Failed LOAD for %s: %s" % (country, e))
    else:
        print("LOAD XML not found for %s" % country)

    # =========================
    # 3. GENERATION
    # =========================
    generation_xml = os.path.join(raw_xml_path, "%s_generation.xml" % country)
    if os.path.exists(generation_xml):
        try:
            print("Converting GENERATION XML -> CSV...")
            parse_xml_generic(
                generation_xml,
                os.path.join(csv_path, "%s_generation.csv" % country),
                "quantity",
                "quantity"
            )
            print("Finish GENERATION")
        except Exception as e:
            print("Failed GENERATION for %s: %s" % (country, e))
    else:
        print("GENERATION XML not found for %s" % country)

    # =========================
    # 4. CROSSBORDER
    # =========================
    print("Converting CROSSBORDER XML -> CSV...")
    neighbours = NEIGHBOURS[country]

    for nb in neighbours:
        # export: country -> nb
        export_xml = os.path.join(raw_xml_path, "%s_to_%s.xml" % (country, nb))
        if os.path.exists(export_xml):
            try:
                parse_xml_generic(
                    export_xml,
                    os.path.join(csv_path, "%s_to_%s.csv" % (country, nb)),
                    "quantity",
                    "quantity"
                )
                print("Finish export:", country, "->", nb)
            except Exception as e:
                print("Failed export %s -> %s: %s" % (country, nb, e))
        else:
            print("Export XML not found:", "%s_to_%s.xml" % (country, nb))

        # import: nb -> country
        import_xml = os.path.join(raw_xml_path, "%s_to_%s.xml" % (nb, country))
        if os.path.exists(import_xml):
            try:
                parse_xml_generic(
                    import_xml,
                    os.path.join(csv_path, "%s_to_%s.csv" % (nb, country)),
                    "quantity",
                    "quantity"
                )
                print("Finish import:", nb, "->", country)
            except Exception as e:
                print("Failed import %s -> %s: %s" % (nb, country, e))
        else:
            print("Import XML not found:", "%s_to_%s.xml" % (nb, country))

    print("Finished:", country)


# =========================
# 主循环
# 遍历全部国家
# =========================
def main():
    all_countries = sorted(NEIGHBOURS.keys())

    for country in all_countries:
        convert_one_country(country)

    print("\nALL COUNTRIES XML -> CSV DONE.")


if __name__ == "__main__":
    main()

接下来一步就是数据的补全
把所有缺时间的数据补全，补上时间戳，全部用N/A代替，不是0，用N/A
普通文件，直接补时间
然后把generation 所有缺失的能源列表补全，不然不方便后面拼在一起，因为有些国家不存在，就是下载不了的
然后统一时间算法
同时不全了缺失的时间
还有一个问题就是有一些PM15,他其实会在数据里插入PM60的数据，所以把PM60全部删除

In [ ]:
import os
import pandas as pd

from entsoe.mappings import NEIGHBOURS, PSRTYPE_MAPPINGS


TZ = "UTC"
START = pd.Timestamp("2024-01-01 00:00", tz=TZ)
END = pd.Timestamp("2024-02-01 00:00", tz=TZ)

PSR_ORDER = ["B%02d" % i for i in range(1, 26)]


def normalize_resolution(resolution):
    if pd.isna(resolution):
        return None

    value = str(resolution).strip()

    mapping = {
        "PT15M": "15min",
        "15min": "15min",
        "15T": "15min",
        "PT30M": "30min",
        "30min": "30min",
        "30T": "30min",
        "PT60M": "1h",
        "PT1H": "1h",
        "60min": "1h",
        "1h": "1h",
        "H": "1h",
    }

    return mapping.get(value, value)


def resolution_to_pandas_freq(resolution):
    normalized = normalize_resolution(resolution)

    mapping = {
        "15min": "15min",
        "30min": "30min",
        "1h": "1h",
    }

    if normalized not in mapping:
        raise ValueError("Unsupported resolution: %s" % resolution)

    return mapping[normalized]


def ensure_datetime(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
    df = df.dropna(subset=["datetime"])
    return df


def choose_best_resolution(df):
    """
    优先级：
    15min > 30min > 1h

    返回：
    filtered_df, best_resolution
    """
    if "resolution" not in df.columns:
        raise ValueError("resolution column not found")

    df = df.copy()
    df["resolution"] = df["resolution"].apply(normalize_resolution)

    available = set(df["resolution"].dropna().tolist())

    if "15min" in available:
        best = "15min"
    elif "30min" in available:
        best = "30min"
    elif "1h" in available:
        best = "1h"
    else:
        raise ValueError("No supported resolution found")

    filtered = df[df["resolution"] == best].copy()
    return filtered, best


def build_full_index_by_resolution(resolution_value):
    freq = resolution_to_pandas_freq(resolution_value)
    return pd.date_range(start=START, end=END, freq=freq, inclusive="left", tz="UTC")


def clean_non_generation_csv(input_path, output_path):
    df = pd.read_csv(input_path)

    if df.empty:
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        return

    # 先处理时间
    df = ensure_datetime(df)

    # 先按时间范围裁切，避免范围外分辨率干扰
    df = df[(df["datetime"] >= START) & (df["datetime"] < END)].copy()

    if df.empty:
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        return

    # 先筛选最佳 resolution
    df, resolution_value = choose_best_resolution(df)

    # 再按时间排序
    df = df.sort_values("datetime").reset_index(drop=True)

    # 再补全时间
    full_index = build_full_index_by_resolution(resolution_value)
    full_df = pd.DataFrame({"datetime": full_index})
    merged = full_df.merge(df, on="datetime", how="left")

    # 填回 resolution
    merged["resolution"] = resolution_value

    merged.to_csv(output_path, index=False, encoding="utf-8-sig")


def clean_generation_csv(input_path, output_path):
    df = pd.read_csv(input_path)

    if df.empty:
        pd.DataFrame(
            columns=[
                "datetime", "quantity", "position", "resolution", "business_type",
                "psr_type", "psr_name", "in_domain", "out_domain",
                "currency_unit", "price_measure_unit"
            ]
        ).to_csv(output_path, index=False, encoding="utf-8-sig")
        return

    # 先处理时间
    df = ensure_datetime(df)

    # 先按时间范围裁切
    df = df[(df["datetime"] >= START) & (df["datetime"] < END)].copy()

    if df.empty:
        pd.DataFrame(
            columns=[
                "datetime", "quantity", "position", "resolution", "business_type",
                "psr_type", "psr_name", "in_domain", "out_domain",
                "currency_unit", "price_measure_unit"
            ]
        ).to_csv(output_path, index=False, encoding="utf-8-sig")
        return

    # 先筛选最佳 resolution
    df, resolution_value = choose_best_resolution(df)

    if "psr_type" not in df.columns:
        df["psr_type"] = "N/A"

    df["psr_type"] = df["psr_type"].fillna("N/A").astype(str).str.strip()
    df.loc[df["psr_type"] == "", "psr_type"] = "N/A"

    # 先按分类顺序 + 时间补全
    full_index = build_full_index_by_resolution(resolution_value)

    full_grid = pd.MultiIndex.from_product(
        [PSR_ORDER, full_index],
        names=["psr_type", "datetime"]
    ).to_frame(index=False)

    merged = full_grid.merge(df, on=["psr_type", "datetime"], how="left")

    merged["resolution"] = resolution_value

    if "business_type" not in merged.columns:
        merged["business_type"] = pd.NA
    if "position" not in merged.columns:
        merged["position"] = pd.NA
    if "in_domain" not in merged.columns:
        merged["in_domain"] = pd.NA
    if "out_domain" not in merged.columns:
        merged["out_domain"] = pd.NA
    if "currency_unit" not in merged.columns:
        merged["currency_unit"] = pd.NA
    if "price_measure_unit" not in merged.columns:
        merged["price_measure_unit"] = pd.NA
    if "quantity" not in merged.columns:
        merged["quantity"] = pd.NA

    merged["psr_name"] = merged["psr_type"].map(PSRTYPE_MAPPINGS).fillna("N/A")

    psr_rank = dict((code, idx) for idx, code in enumerate(PSR_ORDER))
    merged["psr_sort"] = merged["psr_type"].map(psr_rank)

    merged = merged.sort_values(["psr_sort", "datetime"]).reset_index(drop=True)
    merged = merged.drop(columns=["psr_sort"])

    final_cols = [
        "datetime",
        "quantity",
        "position",
        "resolution",
        "business_type",
        "psr_type",
        "psr_name",
        "in_domain",
        "out_domain",
        "currency_unit",
        "price_measure_unit",
    ]

    existing_cols = [c for c in final_cols if c in merged.columns]
    merged = merged[existing_cols]

    merged.to_csv(output_path, index=False, encoding="utf-8-sig")


def clean_one_country(country):
    print("\n==============================")
    print("Processing:", country)
    print("==============================")

    base_path = country
    csv_path = os.path.join(base_path, "csv")
    clean_csv_path = os.path.join(base_path, "clean_csv")

    if not os.path.exists(csv_path):
        print("Skip %s: csv folder not found" % country)
        return

    os.makedirs(clean_csv_path, exist_ok=True)

    # PRICE
    price_csv = os.path.join(csv_path, "%s_price.csv" % country)
    if os.path.exists(price_csv):
        try:
            print("Cleaning PRICE CSV...")
            clean_non_generation_csv(
                price_csv,
                os.path.join(clean_csv_path, "%s_price.csv" % country)
            )
            print("Finish PRICE")
        except Exception as e:
            print("Failed PRICE for %s: %s" % (country, e))
    else:
        print("PRICE CSV not found for %s" % country)

    # LOAD
    load_csv = os.path.join(csv_path, "%s_load.csv" % country)
    if os.path.exists(load_csv):
        try:
            print("Cleaning LOAD CSV...")
            clean_non_generation_csv(
                load_csv,
                os.path.join(clean_csv_path, "%s_load.csv" % country)
            )
            print("Finish LOAD")
        except Exception as e:
            print("Failed LOAD for %s: %s" % (country, e))
    else:
        print("LOAD CSV not found for %s" % country)

    # GENERATION
    generation_csv = os.path.join(csv_path, "%s_generation.csv" % country)
    if os.path.exists(generation_csv):
        try:
            print("Cleaning GENERATION CSV...")
            clean_generation_csv(
                generation_csv,
                os.path.join(clean_csv_path, "%s_generation.csv" % country)
            )
            print("Finish GENERATION")
        except Exception as e:
            print("Failed GENERATION for %s: %s" % (country, e))
    else:
        print("GENERATION CSV not found for %s" % country)

    # CROSSBORDER
    print("Cleaning CROSSBORDER CSV...")
    neighbours = NEIGHBOURS[country]

    for nb in neighbours:
        export_csv = os.path.join(csv_path, "%s_to_%s.csv" % (country, nb))
        if os.path.exists(export_csv):
            try:
                clean_non_generation_csv(
                    export_csv,
                    os.path.join(clean_csv_path, "%s_to_%s.csv" % (country, nb))
                )
                print("Finish export:", country, "->", nb)
            except Exception as e:
                print("Failed export %s -> %s: %s" % (country, nb, e))
        else:
            print("Export CSV not found:", "%s_to_%s.csv" % (country, nb))

        import_csv = os.path.join(csv_path, "%s_to_%s.csv" % (nb, country))
        if os.path.exists(import_csv):
            try:
                clean_non_generation_csv(
                    import_csv,
                    os.path.join(clean_csv_path, "%s_to_%s.csv" % (nb, country))
                )
                print("Finish import:", nb, "->", country)
            except Exception as e:
                print("Failed import %s -> %s: %s" % (nb, country, e))
        else:
            print("Import CSV not found:", "%s_to_%s.csv" % (nb, country))

    print("Finished:", country)


def main():
    all_countries = sorted(NEIGHBOURS.keys())

    for country in all_countries:
        clean_one_country(country)

    print("\nALL COUNTRIES CLEAN CSV DONE.")


if __name__ == "__main__":
    main()

统一时间为1小时

In [ ]:
import os
import pandas as pd

from entsoe.mappings import NEIGHBOURS, PSRTYPE_MAPPINGS


TZ = "UTC"
START = pd.Timestamp("2024-01-01 00:00", tz=TZ)
END = pd.Timestamp("2024-02-01 00:00", tz=TZ)

PSR_ORDER = ["B%02d" % i for i in range(1, 26)]
PSR_NAME_ORDER = [PSRTYPE_MAPPINGS.get(code, code) for code in PSR_ORDER]


def normalize_resolution(resolution):
    if pd.isna(resolution):
        return None

    value = str(resolution).strip()

    mapping = {
        "PT15M": "15min",
        "15min": "15min",
        "15T": "15min",
        "PT30M": "30min",
        "30min": "30min",
        "30T": "30min",
        "PT60M": "1h",
        "PT1H": "1h",
        "60min": "1h",
        "1h": "1h",
        "H": "1h",
    }

    return mapping.get(value, value)


def get_first_valid_resolution(df):
    if "resolution" not in df.columns:
        raise ValueError("resolution column not found")

    valid = df["resolution"].dropna().astype(str).str.strip()
    if valid.empty:
        raise ValueError("No valid resolution found")

    return normalize_resolution(valid.iloc[0])


def ensure_datetime(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
    df = df.dropna(subset=["datetime"])
    return df


def trim_to_target_window(df):
    df = df.copy()
    df = df[(df["datetime"] >= START) & (df["datetime"] < END)].copy()
    return df


def convert_series_to_hourly(df, value_col):
    """
    price / load / crossborder:
    只保留 datetime + value_col
    """
    if df.empty:
        return pd.DataFrame(columns=["datetime", value_col])

    df = ensure_datetime(df)
    df = trim_to_target_window(df)

    resolution_value = get_first_valid_resolution(df)

    keep_cols = ["datetime", value_col]
    df = df[keep_cols].copy()
    df = df.sort_values("datetime").reset_index(drop=True)

    if resolution_value == "1h":
        out = df.copy()
    elif resolution_value == "15min":
        out = (
            df.set_index("datetime")
              .resample("1h", label="left", closed="left")
              .mean()
              .reset_index()
        )
    elif resolution_value == "30min":
        out = (
            df.set_index("datetime")
              .resample("1h", label="left", closed="left")
              .mean()
              .reset_index()
        )
    else:
        raise ValueError("Unsupported resolution for hourly conversion: %s" % resolution_value)

    # 严格控制输出时间范围
    full_hour_index = pd.date_range(start=START, end=END, freq="1h", inclusive="left", tz="UTC")
    full_df = pd.DataFrame({"datetime": full_hour_index})
    out = full_df.merge(out, on="datetime", how="left")

    return out


def convert_generation_to_hourly(df):
    """
    generation:
    先转置成宽表，再统一到 1 小时
    输出列：
    datetime + 各能源名称列
    """
    if df.empty:
        cols = ["datetime"] + PSR_NAME_ORDER
        return pd.DataFrame(columns=cols)

    df = ensure_datetime(df)
    df = trim_to_target_window(df)

    resolution_value = get_first_valid_resolution(df)

    if "psr_type" not in df.columns:
        raise ValueError("psr_type column not found in generation csv")
    if "quantity" not in df.columns:
        raise ValueError("quantity column not found in generation csv")

    df = df.copy()
    df["psr_type"] = df["psr_type"].astype(str).str.strip()
    df["psr_name"] = df["psr_type"].map(PSRTYPE_MAPPINGS).fillna("N/A")

    # 只保留 B01-B25
    df = df[df["psr_type"].isin(PSR_ORDER)].copy()

    # 先按 datetime + psr_name 聚合（防止重复）
    wide = (
        df.groupby(["datetime", "psr_name"], as_index=False)["quantity"]
          .mean()
          .pivot(index="datetime", columns="psr_name", values="quantity")
          .reset_index()
    )

    # 补齐所有 B01-B25 对应的名称列
    for name in PSR_NAME_ORDER:
        if name not in wide.columns:
            wide[name] = pd.NA

    # 列顺序固定
    wide = wide[["datetime"] + PSR_NAME_ORDER]

    wide = wide.sort_values("datetime").reset_index(drop=True)

    if resolution_value == "1h":
        out = wide.copy()
    elif resolution_value == "15min":
        out = (
            wide.set_index("datetime")
                .resample("1h", label="left", closed="left")
                .mean()
                .reset_index()
        )
    elif resolution_value == "30min":
        out = (
            wide.set_index("datetime")
                .resample("1h", label="left", closed="left")
                .mean()
                .reset_index()
        )
    else:
        raise ValueError("Unsupported resolution for generation hourly conversion: %s" % resolution_value)

    # 最终严格补齐到完整小时轴
    full_hour_index = pd.date_range(start=START, end=END, freq="1h", inclusive="left", tz="UTC")
    full_df = pd.DataFrame({"datetime": full_hour_index})
    out = full_df.merge(out, on="datetime", how="left")

    # 再保证列顺序
    out = out[["datetime"] + PSR_NAME_ORDER]

    return out


def process_one_country(country):
    print("\n==============================")
    print("Processing:", country)
    print("==============================")

    base_path = country
    clean_csv_path = os.path.join(base_path, "clean_csv")
    hourly_csv_path = os.path.join(base_path, "hourly_csv")

    if not os.path.exists(clean_csv_path):
        print("Skip %s: clean_csv folder not found" % country)
        return

    os.makedirs(hourly_csv_path, exist_ok=True)

    # PRICE
    price_csv = os.path.join(clean_csv_path, "%s_price.csv" % country)
    if os.path.exists(price_csv):
        try:
            print("Converting PRICE to hourly...")
            df = pd.read_csv(price_csv)
            out = convert_series_to_hourly(df, "price")
            out.to_csv(
                os.path.join(hourly_csv_path, "%s_price.csv" % country),
                index=False,
                encoding="utf-8-sig"
            )
            print("Finish PRICE")
        except Exception as e:
            print("Failed PRICE for %s: %s" % (country, e))
    else:
        print("PRICE CSV not found for %s" % country)

    # LOAD
    load_csv = os.path.join(clean_csv_path, "%s_load.csv" % country)
    if os.path.exists(load_csv):
        try:
            print("Converting LOAD to hourly...")
            df = pd.read_csv(load_csv)
            out = convert_series_to_hourly(df, "quantity")
            out.to_csv(
                os.path.join(hourly_csv_path, "%s_load.csv" % country),
                index=False,
                encoding="utf-8-sig"
            )
            print("Finish LOAD")
        except Exception as e:
            print("Failed LOAD for %s: %s" % (country, e))
    else:
        print("LOAD CSV not found for %s" % country)

    # GENERATION
    generation_csv = os.path.join(clean_csv_path, "%s_generation.csv" % country)
    if os.path.exists(generation_csv):
        try:
            print("Converting GENERATION to hourly...")
            df = pd.read_csv(generation_csv)
            out = convert_generation_to_hourly(df)
            out.to_csv(
                os.path.join(hourly_csv_path, "%s_generation.csv" % country),
                index=False,
                encoding="utf-8-sig"
            )
            print("Finish GENERATION")
        except Exception as e:
            print("Failed GENERATION for %s: %s" % (country, e))
    else:
        print("GENERATION CSV not found for %s" % country)

    # CROSSBORDER
    print("Converting CROSSBORDER to hourly...")
    neighbours = NEIGHBOURS[country]

    for nb in neighbours:
        export_csv = os.path.join(clean_csv_path, "%s_to_%s.csv" % (country, nb))
        if os.path.exists(export_csv):
            try:
                df = pd.read_csv(export_csv)
                out = convert_series_to_hourly(df, "quantity")
                out.to_csv(
                    os.path.join(hourly_csv_path, "%s_to_%s.csv" % (country, nb)),
                    index=False,
                    encoding="utf-8-sig"
                )
                print("Finish export:", country, "->", nb)
            except Exception as e:
                print("Failed export %s -> %s: %s" % (country, nb, e))
        else:
            print("Export CSV not found:", "%s_to_%s.csv" % (country, nb))

        import_csv = os.path.join(clean_csv_path, "%s_to_%s.csv" % (nb, country))
        if os.path.exists(import_csv):
            try:
                df = pd.read_csv(import_csv)
                out = convert_series_to_hourly(df, "quantity")
                out.to_csv(
                    os.path.join(hourly_csv_path, "%s_to_%s.csv" % (nb, country)),
                    index=False,
                    encoding="utf-8-sig"
                )
                print("Finish import:", nb, "->", country)
            except Exception as e:
                print("Failed import %s -> %s: %s" % (nb, country, e))
        else:
            print("Import CSV not found:", "%s_to_%s.csv" % (nb, country))

    print("Finished:", country)


def main():
    all_countries = sorted(NEIGHBOURS.keys())

    for country in all_countries:
        process_one_country(country)

    print("\nALL COUNTRIES HOURLY CSV DONE.")


if __name__ == "__main__":
    main()

倒数第二部，根据price时间，合并。把load,price, generation, export, import 全部合并起来

In [ ]:
import os
import pandas as pd
from entsoe.mappings import NEIGHBOURS


TZ = "UTC"


def read_csv_safe(path):
    if not os.path.exists(path):
        return None
    return pd.read_csv(path, low_memory=False)


def merge_one_country(country):
    print("\n==============================")
    print("Merging:", country)
    print("==============================")

    base_path = country
    hourly_path = os.path.join(base_path, "hourly_csv")

    if not os.path.exists(hourly_path):
        print("Skip:", country, "no hourly_csv folder")
        return

    # =========================
    # 1. PRICE（主表）
    # =========================
    price_path = os.path.join(hourly_path, f"{country}_price.csv")
    price_df = read_csv_safe(price_path)

    if price_df is None:
        print("Skip:", country, "no price file")
        return

    price_df["datetime"] = pd.to_datetime(price_df["datetime"], utc=True)
    price_df = price_df.rename(columns={"datetime": "time"})

    # =========================
    # 初始化主表
    # =========================
    df = price_df.copy()

    # =========================
    # 2. LOAD
    # =========================
    load_path = os.path.join(hourly_path, f"{country}_load.csv")
    load_df = read_csv_safe(load_path)

    if load_df is not None:
        load_df["datetime"] = pd.to_datetime(load_df["datetime"], utc=True)
        load_df = load_df.rename(columns={"datetime": "time", "quantity": "load"})
        df = df.merge(load_df[["time", "load"]], on="time", how="left")

    # =========================
    # 3. GENERATION
    # =========================
    gen_path = os.path.join(hourly_path, f"{country}_generation.csv")
    gen_df = read_csv_safe(gen_path)

    if gen_df is not None:
        gen_df["datetime"] = pd.to_datetime(gen_df["datetime"], utc=True)
        gen_df = gen_df.rename(columns={"datetime": "time"})
        df = df.merge(gen_df, on="time", how="left")

    # =========================
    # 4. EXPORT / IMPORT
    # =========================
    neighbours = NEIGHBOURS[country]

    export_cols = []
    import_cols = []

    for nb in neighbours:

        # ===== export =====
        export_path = os.path.join(hourly_path, f"{country}_to_{nb}.csv")
        exp_df = read_csv_safe(export_path)

        if exp_df is not None:
            exp_df["datetime"] = pd.to_datetime(exp_df["datetime"], utc=True)

            col_name = f"export to {nb}"
            exp_df = exp_df.rename(columns={"datetime": "time", "quantity": col_name})

            df = df.merge(exp_df[["time", col_name]], on="time", how="left")
            export_cols.append(col_name)

        # ===== import =====
        import_path = os.path.join(hourly_path, f"{nb}_to_{country}.csv")
        imp_df = read_csv_safe(import_path)

        if imp_df is not None:
            imp_df["datetime"] = pd.to_datetime(imp_df["datetime"], utc=True)

            col_name = f"import from {nb}"
            imp_df = imp_df.rename(columns={"datetime": "time", "quantity": col_name})

            df = df.merge(imp_df[["time", col_name]], on="time", how="left")
            import_cols.append(col_name)

    # =========================
    # 5. total export / import
    # =========================
    if export_cols:
        df["total export"] = df[export_cols].sum(axis=1, skipna=True)
    else:
        df["total export"] = pd.NA

    if import_cols:
        df["total import"] = df[import_cols].sum(axis=1, skipna=True)
    else:
        df["total import"] = pd.NA

    # =========================
    # 6. 插入 country / time_zone
    # =========================
    df.insert(1, "country", country)
    df.insert(2, "time_zone", TZ)

    # =========================
    # 7. 列顺序整理
    # =========================
    base_cols = ["time", "country", "time_zone", "price", "load"]

    # generation列（排除已有列）
    gen_cols = [c for c in df.columns if c not in base_cols and
                not c.startswith("export") and
                not c.startswith("import") and
                c not in ["total export", "total import"]]

    final_cols = (
        base_cols
        + gen_cols
        + export_cols
        + import_cols
        + ["total export", "total import"]
    )

    df = df[final_cols]

    # =========================
    # 8. 输出
    # =========================
    output_path = os.path.join(base_path, f"{country}.csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("Finished:", country)


def main():
    all_countries = sorted(NEIGHBOURS.keys())

    for country in all_countries:
        merge_one_country(country)

    print("\nALL COUNTRIES MERGED DONE.")


if __name__ == "__main__":
    main()

最后一步，删除各国家进出口，只保留total, 合并成超极长的表格，删除了时间的—+0000啥的

In [ ]:
import os
import pandas as pd
from entsoe.mappings import NEIGHBOURS


OUTPUT_FILE = "ALL_COUNTRIES.csv"


def clean_time_column(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True)
    dt = dt.dt.tz_localize(None)
    return dt.dt.strftime("%Y-%m-%d %H:%M:%S")


def load_one_country_file(country):
    """
    读取每个国家最终生成好的文件：
    ./{country}/{country}.csv
    """
    path = os.path.join(country, f"{country}.csv")

    if not os.path.exists(path):
        print("Missing:", path)
        return None

    df = pd.read_csv(path, low_memory=False)

    if "time" not in df.columns:
        print("Skip:", path, "because 'time' column not found")
        return None

    # 时间转成干净格式，去掉 +00:00
    df["time"] = clean_time_column(df["time"])

    # 删除所有 export / import 明细列
    drop_cols = [
        c for c in df.columns
        if c.startswith("export to ") or c.startswith("import from ")
    ]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    return df


def main():
    all_countries = sorted(NEIGHBOURS.keys())
    all_dfs = []

    print("Start merging all country files...")

    for country in all_countries:
        print("Loading:", country)
        df = load_one_country_file(country)
        if df is not None:
            all_dfs.append(df)

    if not all_dfs:
        print("No country files found. Stop.")
        return

    # 纵向合并
    merged = pd.concat(all_dfs, axis=0, ignore_index=True, sort=False)

    # 调整列顺序
    front_cols = ["time", "country", "time_zone", "price", "load"]
    end_cols = ["total export", "total import"]

    middle_cols = [
        c for c in merged.columns
        if c not in front_cols + end_cols
    ]

    final_cols = (
        [c for c in front_cols if c in merged.columns]
        + middle_cols
        + [c for c in end_cols if c in merged.columns]
    )

    merged = merged[final_cols]

    # 保存
    merged.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("Done.")
    print("Output file:", OUTPUT_FILE)
    print("Rows:", len(merged))
    print("Columns:", len(merged.columns))


if __name__ == "__main__":
    main()